# Módulo 3: Agentes de IA
- **Sesión:** 01 — Fundamentos de agentes de IA
- **Tema:** Patrones de razonamiento  — ReAct y Plan-and-Execute
- **Fecha:** 20/10/2026

## Fundamentos — Patrones de razonamiento

| Patrón | Cuándo decide | Ciclo |
|---|---|---|
| **ReAct** | Durante la tarea, un paso a la vez | razonar → actuar → observar |
| **Plan-and-Execute** | Antes de empezar: escribe el plan completo | planificar → ejecutar → sintetizar |

In [39]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
import os

In [40]:

# Generar una funcion que obtenga el modelo LLM

def get_llm(prefix_model, temperature=0.9):
    """
    Obtiene un modelo de lenguaje (LLM) basado en el modelo especificado.

    Args:
        model (str): El nombre del modelo a utilizar.
        temperature (float, optional): La temperatura para la generación de texto. 
                                        Valores más altos producen respuestas más creativas.
                                        Valores más bajos producen respuestas más deterministas.
                                        Por defecto es 0.9.

    Returns:
        llm: Un objeto de modelo de lenguaje inicializado.
    """
    llm = init_chat_model(
                            model=prefix_model, 
                            # temperature=temperature
                          )
    return llm

# Configuración del modelo de lenguaje
load_dotenv(override = True)

prefix_model = os.getenv("PREFIX_MODEL")

# Obtención del modelo
llm = get_llm(prefix_model)



In [41]:
from langchain.tools import tool

# Datos simulados de TiendaYa

## Estado, Precio, Costo de Envio
PEDIDOS = {
    "48213": {"estado": "entregado", "monto": 320.00, "envio": 15.00},
    "77100": {"estado": "en ruta", "monto": 500.00, "envio": 10.00},
    "50555": {"estado": "en preparación", "monto": 1250.00, "envio": 0.00},
}

POLITICAS = {
    "reembolsos": ("El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. "
                   "El envío se devuelve solo si el producto llegó dañado."),
    "devoluciones": "15 días calendario desde la entrega; 7 días para productos electrónicos.",
}


@tool
def consultar_pedido(numero_pedido: str) -> str:
    """
        Devuelve el estado, el monto de los productos y el costo de envío de un pedido de TiendaYa.

    """
    p = PEDIDOS.get(numero_pedido)
    if p is None:
        return f"No existe el pedido {numero_pedido}."

    return f"Pedido {numero_pedido}: {p['estado']}, productos S/ {p['monto']:.2f}, envío S/ {p['envio']:.2f}."


@tool
def consultar_politica(tema: str) -> str:
    """
        Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'reembolsos' .
    """
    return POLITICAS.get(tema.lower(), "No existe una política sobre ese tema.")


@tool
def calcular_reembolso(monto_productos: float, costo_envio: float, incluir_envio: bool) -> str:
    """
        Calcula el monto a reembolsar. Suma el envío solo si incluir_envio es verdadero.
    """
    
    total = monto_productos + (costo_envio if incluir_envio else 0)
    
    return f"Monto a reembolsar: S/ {total:.2f}"


print(consultar_pedido.invoke(
                                {
                                    "numero_pedido": "48213"
                                }
                            ))

print(consultar_politica.invoke({"tema": "reembolsos"}))

print(calcular_reembolso.invoke({"monto_productos": 320, "costo_envio": 15, "incluir_envio": True}))




Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.
El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. El envío se devuelve solo si el producto llegó dañado.
Monto a reembolsar: S/ 335.00


## 1 — ReAct: razonar, actuar, observar


In [42]:
CONSULTA = "Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?"

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

SYSTEM_PROMPT="Eres el asistente de TiendaYa. Usa las herramientas para obtener los datos; no los inventes.",

agente = create_agent(
    model=llm,
    tools=[ consultar_pedido, 
            consultar_politica,
            calcular_reembolso],
    system_prompt=SYSTEM_PROMPT
)

# Mensaje
mensaje = {
            "messages": [{
                            "role": "user", 
                            "content": CONSULTA}]
        }

resultado = agente.invoke(mensaje)

print ("------------------------")

'''
for m in resultado["messages"]:
    if getattr(m, "tool_calls", None):
        contenido = "solicita " + ", ".join(f"{t['name']}({t['args']})" for t in m.tool_calls)
    else:
        contenido = m.text
    print(f"[{type(m).__name__:12}] {contenido[:150]}")
#'''

print ("------------------------")

ite = 0
for m in resultado["messages"]:
    if isinstance(m,AIMessage) and m.tool_calls :
        ite += 1
        for tool in m.tool_calls:
            print(f"Ite {ite} | ACTUAR -> {tool['name']}({tool['args']}")
    elif isinstance(m,ToolMessage) :
        print( f"Ite {ite} | OBSERVAR -> {m.content}")

print(resultado["messages"][-1].text)

------------------------
------------------------
Ite 1 | ACTUAR -> consultar_pedido({'numero_pedido': '48213'}
Ite 1 | ACTUAR -> consultar_politica({'tema': 'reembolsos'}
Ite 1 | OBSERVAR -> Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.
Ite 1 | OBSERVAR -> El reembolso se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. El envío se devuelve solo si el producto llegó dañado.
Ite 2 | ACTUAR -> calcular_reembolso({'costo_envio': 15, 'incluir_envio': True, 'monto_productos': 320}
Ite 2 | OBSERVAR -> Monto a reembolsar: S/ 335.00
El monto total a devolver es de **S/ 335.00** (que incluye S/ 320.00 por los productos y S/ 15.00 por el costo de envío, ya que el producto llegó dañado). 

El reembolso se abonará al mismo medio de pago que utilizaste en la compra y tardará entre **5 y 10 días hábiles** en reflejarse.


## 2 — Plan-and-Execute: primero el plan, luego la ejecución

![Plan-and-Execute](graficos/M3_S01_razonamiento_paso2_plan_and_execute.png)


In [43]:
# 

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

CONSULTA = "Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?"


HERRAMIENTAS = [ consultar_pedido, 
                 consultar_politica,
                 calcular_reembolso]


NOMBRES_HERRAMIENTAS = [h.name for h in HERRAMIENTAS]

'''
NOMBRES_HERRAMIENTAS = ['consultar_pedido', 
                        'consultar_politica', 
                        'calcular_reembolso']
'''

DESCRIPCION_HERRAMIENTAS = "\n".join(f"- {h.name}: {h.description}" for h in HERRAMIENTAS)

'''
DESCRIPCION_HERRAMIENTAS = ['- consultar_pedido: Devuelve el estado, el monto de los productos y el costo de envío de un pedido de TiendaYa.', 
                            '- consultar_politica: Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'reembolsos' .', 
                            '- calcular_reembolso: Calcula el monto a reembolsar. Suma el envío solo si incluir_envio es verdadero.]
'''


#print(HERRAMIENTAS)
print(NOMBRES_HERRAMIENTAS)
print(DESCRIPCION_HERRAMIENTAS)


['consultar_pedido', 'consultar_politica', 'calcular_reembolso']
- consultar_pedido: Devuelve el estado, el monto de los productos y el costo de envío de un pedido de TiendaYa.
- consultar_politica: Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'reembolsos' .
- calcular_reembolso: Calcula el monto a reembolsar. Suma el envío solo si incluir_envio es verdadero.


In [44]:

# -----------------------
# 1. PLANIFICAR
# -----------------------

PROMTP_PLAN = f"""
        
    Herramientas disponibles: 
    {DESCRIPCION_HERRAMIENTAS}
    Escribe un plan para atender la consulta del cliente. Cada paso debe usar UNA de las herramientas
    e indicar su nombre. No incluyas pasos que las herramientas no puedan realizar.

    Responde solo con una lista numerada, un paso por línea.
    
       Consulta: {CONSULTA}
    """ 

print(f"PROMPT PLANIFICACION {PROMTP_PLAN}")

texto = llm.invoke(PROMTP_PLAN).text

print(texto)

# Se conservan solo las líneas que empiezan con un número
plan = [linea.strip() for linea in texto.splitlines() if linea.strip()[:1].isdigit()]

print("PLAN:")
for paso in plan:
    print("  ", paso)


PROMPT PLANIFICACION 

    Herramientas disponibles: 
    - consultar_pedido: Devuelve el estado, el monto de los productos y el costo de envío de un pedido de TiendaYa.
- consultar_politica: Devuelve la política de TiendaYa sobre un tema: 'devoluciones' o 'reembolsos' .
- calcular_reembolso: Calcula el monto a reembolsar. Suma el envío solo si incluir_envio es verdadero.
    Escribe un plan para atender la consulta del cliente. Cada paso debe usar UNA de las herramientas
    e indicar su nombre. No incluyas pasos que las herramientas no puedan realizar.

    Responde solo con una lista numerada, un paso por línea.

       Consulta: Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?
    
1. consultar_pedido(48213)
2. consultar_politica('reembolsos')
3. calcular_reembolso(...)
PLAN:
   1. consultar_pedido(48213)
   2. consultar_politica('reembolsos')
   3. calcular_reembolso(...)


In [45]:
# -----------------------
# 2. REVISAR
# -----------------------

#aprobados = [paso for paso in plan if any(nombre in paso for nombre in NOMBRES_HERRAMIENTAS)]

aprobados = []

for paso in plan:
    # Revisamos si este paso menciona alguna herramienta
    menciona_herramienta = False

    for nombre in NOMBRES_HERRAMIENTAS:
        if nombre in paso:
            menciona_herramienta = True
            break  # Ya encontramos una, no hace falta seguir buscando

    if menciona_herramienta:
        aprobados.append(paso)

print(f"APROBADOS: {aprobados}")

for paso in plan:
    if paso not in aprobados:
        print("  DESCARTADO:", paso)


APROBADOS: ['1. consultar_pedido(48213)', "2. consultar_politica('reembolsos')", '3. calcular_reembolso(...)']


In [46]:
# -----------------------
# 3. EJECUTAR
# -----------------------

SYSTEM_PROMPT="Resuelve únicamente el paso indicado, usando las herramientas, y devuelve el resultado.",

# El ejecutor es un agente que resuelve UN paso a la vez
ejecutor = create_agent(
    model=llm,
    tools=HERRAMIENTAS,
    system_prompt=SYSTEM_PROMPT
)

resultados = []
for paso in plan:
    # 1. Juntamos todo lo que ya se resolvió, un resultado por línea
    resultados_previos = "\n".join(resultados)

    # 2. Armamos el mensaje que recibirá el agente
    mensaje = (
        f""" 
           Consulta del cliente: {CONSULTA}

           Resultados anteriores:
                   {resultados_previos}

           Paso a resolver: {paso}
        """
    )

    # 3. Le pedimos al agente que resuelva este paso
    respuesta = ejecutor.invoke({
        "messages": [{
                      "role": "user", 
                      "content": mensaje}]
    })

    # 4. Tomamos el último mensaje de la conversación (la respuesta final del agente)
    texto_respuesta = respuesta["messages"][-1].text

    # 5. Guardamos el resultado para que lo vean los pasos siguientes
    resultados.append(f"{paso} -> {texto_respuesta}")

    # 6. Mostramos un resumen (solo los primeros 150 caracteres)
    print("\nEJECUTADO:", resultados[-1][:150])



EJECUTADO: 1. consultar_pedido(48213) -> Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.

EJECUTADO: 2. consultar_politica('reembolsos') -> La política de reembolsos indica que el dinero se abona al mismo medio de pago usado en la compra y tarda de 5 

EJECUTADO: 3. calcular_reembolso(...) -> El monto a reembolsar es de S/ 335.00 (S/ 320.00 por los productos más S/ 15.00 por el costo de envío).


In [47]:
# -----------------------
# 4. SINTETIZAR ( HUMANIZAR)
# -----------------------

PROMPT_SINTETIZAR = f""" 
    Redacta una respuesta breve y cordial para el cliente usando SOLO estos resultados.
    No agregues datos que no estén en ellos.

    Consulta: {CONSULTA}
    Resultados:
       {"\n".join(resultados)}
"""

print(PROMPT_SINTETIZAR)

respuesta = llm.invoke(PROMPT_SINTETIZAR).text

print(f"RESPUESTA: {respuesta}")

 
    Redacta una respuesta breve y cordial para el cliente usando SOLO estos resultados.
    No agregues datos que no estén en ellos.

    Consulta: Mi pedido 48213 llegó dañado. ¿Cuánto me devuelven y en cuántos días?
    Resultados:
       1. consultar_pedido(48213) -> Pedido 48213: entregado, productos S/ 320.00, envío S/ 15.00.
2. consultar_politica('reembolsos') -> La política de reembolsos indica que el dinero se abona al mismo medio de pago usado en la compra y tarda de 5 a 10 días hábiles. Además, el costo del envío se devuelve porque el producto llegó dañado.
3. calcular_reembolso(...) -> El monto a reembolsar es de S/ 335.00 (S/ 320.00 por los productos más S/ 15.00 por el costo de envío).

RESPUESTA: ¡Hola! Lamentamos que tu pedido 48213 haya llegado dañado. El monto total a reembolsar es de S/ 335.00 (S/ 320.00 por los productos más S/ 15.00 por el costo de envío). El dinero se abonará al mismo medio de pago usado en la compra y tardará de 5 a 10 días hábiles.


- Ejercicio 1 : Cada paso del Patrón Plan and Execute ,mejorarlo usando funciones

- Ejercicio 2 : Se desea incluir el paso de devolucion ( registrar_devolucion) , el paso debe ejecutarse despues de consultar el pedido, se necesita el numero del pedido, en la respuesta se debe mencionar el numero de ticket 

```python
@tool
def registrar_devolucion(numero_pedido: str) -> str:
    """Registra la devolución de un pedido de TiendaYa y devuelve el código del ticket."""
    return f"Devolución registrada con el ticket DEV-{numero_pedido}. Lleve el producto a cualquier agencia en los próximos 3 días."
```